# NLP ESG KPI Extraction — Demo

Runs ingest → index → both extractors → comparison table → evaluation, end to end. All logic lives in `src/nlp_esg/`; this notebook is a thin driver.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from nlp_esg.pipeline import (
    load_indexed_reports, run_extraction, load_gold_labels, run_evaluation,
)
from nlp_esg.compare import build_comparison_table

indexed = load_indexed_reports()
print(f'Loaded {len(indexed)} reports')

In [ ]:
extractions = run_extraction(indexed, include_llm=True)
print(f'Produced {len(extractions)} extractions')

## Comparison table — Baseline

In [ ]:
build_comparison_table(extractions, extractor='baseline')

## Comparison table — LLM

In [ ]:
build_comparison_table(extractions, extractor='llm')

## Evaluation — P / R / F1 / coverage

In [ ]:
golds = load_gold_labels()
metrics = run_evaluation(extractions, golds)
metrics

## Embedding-model comparison: MiniLM vs ClimateBERT

Re-run the baseline with MiniLM to compare retrieval quality.

In [ ]:
import os
os.environ['EMBEDDING_MODEL'] = 'minilm'
# Reload modules so the new env var takes effect. config caches EMBEDDING_MODEL_NAME
# at import time, so it must be reloaded before retrieval/pipeline.
import importlib, nlp_esg.config, nlp_esg.retrieval, nlp_esg.pipeline
importlib.reload(nlp_esg.config)
importlib.reload(nlp_esg.retrieval)
importlib.reload(nlp_esg.pipeline)
from nlp_esg.pipeline import load_indexed_reports as load2, run_extraction as run2, run_evaluation as eval2
indexed_mini = load2()
extractions_mini = run2(indexed_mini, include_llm=False)
eval2(extractions_mini, golds)

## Qualitative cases

Manually inspect cells where one extractor wins and the other doesn't. Fill in after running against real data.

- **Baseline wins:** ...
- **LLM wins:** ...
- **Both fail:** ...